In [1]:
import duckdb
con = duckdb.connect("../data/processed/agro.duckdb", read_only=True)

In [2]:
con.sql("""
SELECT period,
       ROUND(yield_t_ha, 2) AS yield_t_ha,
       ROUND(yield_t_ha - LAG(yield_t_ha) OVER (ORDER BY period), 2) AS change_vs_prev
FROM yields
WHERE region = 'Casanare'
  AND period >= '2008-I'
ORDER BY period
""").df()

,period,yield_t_ha,change_vs_prev
0,2008-I,6.08,NaN
1,2008-II,5.40,-0.68
2,2009-I,6.02,0.62
3,2009-II,5.11,-0.91
4,2010-I,5.36,0.24
5,2010-II,5.39,0.04
6,2011-I,5.06,-0.33
7,2011-II,4.26,-0.80
8,2012-I,4.89,0.64
9,2012-II,4.88,-0.01


Semester I almost always rises vs. the previous semester II, and semester II almost always falls vs. semester I (exception: 2011-I fell).

In [11]:
con.sql(""" SELECT period,
       ROUND(yield_t_ha, 2) AS yield_t_ha,
       ROUND(AVG(yield_t_ha) OVER (ORDER BY period ROWS BETWEEN 5 PRECEDING AND CURRENT ROW), 2) AS moving_avg_6
FROM yields
WHERE region = 'Casanare'
  AND period >= '2008-I'
ORDER BY period
""").df()

,period,yield_t_ha,moving_avg_6
0,2008-I,6.08,6.08
1,2008-II,5.40,5.74
2,2009-I,6.02,5.83
3,2009-II,5.11,5.65
4,2010-I,5.36,5.60
5,2010-II,5.39,5.56
6,2011-I,5.06,5.39
7,2011-II,4.26,5.20
8,2012-I,4.89,5.01
9,2012-II,4.88,4.97


In [17]:
con.sql("""
WITH casanare AS (
    SELECT period,
           split_part(period, '-', 2) AS semester,
           yield_t_ha
    FROM yields
    WHERE region = 'Casanare'
      AND period >= '2008-I'
)
SELECT semester,
       COUNT(*)                     AS semesters,
       ROUND(AVG(yield_t_ha), 2)    AS avg_yield,
       ROUND(STDDEV(yield_t_ha), 2) AS std_yield,
       ROUND(min(yield_t_ha), 2) AS min_yield,
       ROUND(max(yield_t_ha), 2) AS max_yield
FROM casanare
GROUP BY semester
ORDER BY semester
""").df()

,semester,semesters,avg_yield,std_yield,min_yield,max_yield
0,I,19,5.63,0.37,4.89,6.08
1,II,18,5.12,0.34,4.26,5.46


In [20]:
con.sql("""
WITH casanare AS (
    SELECT period,
           split_part(period, '-', 2) AS semester,
           yield_t_ha
    FROM yields
    WHERE region = 'Casanare'
      AND period >= '2008-I'
)
SELECT period,
       semester,
       ROUND(yield_t_ha, 2) AS yield_t_ha,
       ROUND(yield_t_ha - LAG(yield_t_ha) OVER (ORDER BY period), 2) AS change_vs_prev,
       ROUND(yield_t_ha - LAG(yield_t_ha) OVER (PARTITION BY semester ORDER BY period), 2) AS change_vs_last_year
FROM casanare
ORDER BY period
""").df()


,period,semester,yield_t_ha,change_vs_prev,change_vs_last_year
0,2008-I,I,6.08,NaN,NaN
1,2008-II,II,5.40,-0.68,NaN
2,2009-I,I,6.02,0.62,-0.06
3,2009-II,II,5.11,-0.91,-0.29
4,2010-I,I,5.36,0.24,-0.66
5,2010-II,II,5.39,0.04,0.28
6,2011-I,I,5.06,-0.33,-0.30
7,2011-II,II,4.26,-0.80,-1.14
8,2012-I,I,4.89,0.64,-0.17
9,2012-II,II,4.88,-0.01,0.63


In [21]:
con.sql("""
WITH casanare AS (
    SELECT period,
           split_part(period, '-', 2) AS semester,
           yield_t_ha
    FROM yields
    WHERE region = 'Casanare'
      AND period >= '2008-I'
)
SELECT period,
       semester,
       ROUND(yield_t_ha, 2) AS yield_t_ha,
       ROUND(yield_t_ha / AVG(yield_t_ha) OVER (), 3)                      AS ratio_all,
       ROUND(yield_t_ha / AVG(yield_t_ha) OVER (PARTITION BY semester), 3) AS ratio_own_semester
FROM casanare
ORDER BY ratio_all""").df()

,period,semester,yield_t_ha,ratio_all,ratio_own_semester
0,2011-II,II,4.26,0.790,0.831
1,2013-II,II,4.54,0.844,0.887
2,2017-II,II,4.67,0.867,0.911
3,2012-II,II,4.88,0.907,0.953
4,2012-I,I,4.89,0.909,0.869
5,2015-II,II,5.00,0.929,0.976
6,2011-I,I,5.06,0.940,0.899
7,2013-I,I,5.07,0.941,0.900
8,2009-II,II,5.11,0.950,0.998
9,2023-II,II,5.15,0.956,1.005


### Findings

1. Semester I yields more than semester II: 5.63 vs. 5.12 t/ha on average, about 9% less in semester II. Comparing each semester with the previous one mixes this seasonal gap with real losses, so semesters I and II must be analyzed separately.
2. Against the overall average, 2011-II and 2013-II fall below 85%. Against the semester II average, only 2011-II does: 2013-II looked like a bad semester only because it is a semester II.